# Nemotron LoRA — train on Google Colab (A100)

**Requires Colab Pro (A100, 40 GB).** We use the **8-bit** path because the 31.6B model is ~63 GB in bf16 and won't fit in 40 GB.

**First:** Runtime → Change runtime type → **A100 GPU** → Save. Then run the cells top to bottom. As soon as training finishes, run the *Save adapter* cell (Colab sessions can drop). Packaging + submission happen on Kaggle afterwards.

## 0. Confirm you actually got an A100 (~40 GB)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Code + dependencies

In [ ]:
!git clone -b build/nemotron-pipeline https://github.com/SebAustin/NVIDIA-Nemotron-Model-Reasoning-Challenge repo
%cd repo
!pip install -q "transformers>=4.45,<5" peft trl datasets accelerate bitsandbytes psutil hf_transfer

## 2. mamba_ssm + causal_conv1d (wheels matching this torch)

In [ ]:
!python kaggle_wheels/fetch_torch_locked_wheels.py --dest /tmp/mw
!pip install -q --no-deps /tmp/mw/causal_conv1d-*.whl /tmp/mw/mamba_ssm-*.whl
import importlib
for m in ("mamba_ssm","causal_conv1d"):
    try: importlib.import_module(m); print('ok', m)
    except Exception as e: print('FAIL', m, '->', repr(e)[:120])

## 3. Hugging Face login
The base model may be gated — accept its license on the model's HF page first, then paste an HF token below.

In [ ]:
import os
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'   # faster ~60GB download
from huggingface_hub import login
login()

## 4. Competition data (your Kaggle API token)
Paste your Kaggle **KGAT** token below. Do **not** commit it anywhere.

In [ ]:
import os, urllib.request
TOK = "KGAT_xxxxxxxxxxxxxxxx"   # <-- paste your Kaggle API token
os.makedirs('data', exist_ok=True)
url = 'https://www.kaggle.com/api/v1/competitions/data/download/nvidia-nemotron-model-reasoning-challenge/train.csv'
req = urllib.request.Request(url, headers={'Authorization': f'Bearer {TOK}'})
open('data/train.csv','wb').write(urllib.request.urlopen(req).read())
print('train.csv:', os.path.getsize('data/train.csv'), 'bytes')

## 5. Build the SFT data (no GPU, ~1 min)

In [ ]:
!python scripts/01_eda.py --data-dir data
!python scripts/02_prepare_data.py --data-dir data

## 6. Train (8-bit, single 40 GB A100)
Runs a smoke test first to confirm it fits, then trains. If it OOMs, lower `SFT_MAX_SEQ_LENGTH` to 768 / `LORA_R` to 8, or use Modal (last cell).

In [ ]:
import os
os.environ['NEMOTRON_MAX_MEMORY_GPU'] = '38GiB'   # keep weights on the GPU
os.environ['SFT_MAX_SEQ_LENGTH']      = '1024'     # data is short; saves memory
!python scripts/03_train_lora.py --data-path data/train_sft.jsonl --output-dir lora_adapter

## 7. Save the adapter OFF Colab (do this immediately)

In [ ]:
import shutil
shutil.make_archive('/content/lora_adapter', 'zip', 'lora_adapter')
from google.colab import files; files.download('/content/lora_adapter.zip')

## 8. Next: package + submit on Kaggle
Upload `lora_adapter` as a Kaggle **dataset**, then run `kaggle_package_submit.ipynb` in a Kaggle notebook → Save Version → Submit.

---

**Simpler alternative — Modal (80 GB A100, bf16):** instead of cells 0–7,
```
!pip install -q modal && modal setup
!modal secret create nemotron KAGGLE_TOKEN=KGAT_xxx
!cd repo && modal run modal/train_modal.py && modal volume get nemotron-out lora_adapter ./lora_adapter
```